# Trace Analysis

In [7]:
import json
from pathlib import Path

import pandas as pd
import ipywidgets as widgets
from IPython.display import display

from trace_viewer import show_trace

TRACES_DIR = Path("../traces")
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 50)

## Dashboard

In [8]:
results = pd.read_csv(TRACES_DIR / "results.csv")
results["passed"] = results["passed"].astype(bool)
print(f"{len(results)} total results across {results['timestamp'].nunique()} runs")

113 total results across 8 runs


In [9]:
# Run-over-run summary (last 10 runs)
runs = results.groupby("timestamp").agg(
    passed=("passed", "sum"),
    questions=("passed", "count"),
    pass_rate=("passed", "mean"),
    avg_tool_calls=("tool_calls", "mean"),
    avg_turn_latency_s=("avg_turn_latency_s", "mean"),
    avg_tokens=("total_tokens", "mean"),
    avg_cost=("cost_usd", "mean"),
    total_cost=("cost_usd", "sum"),
).tail(10)
runs["pass_rate"] = runs["pass_rate"].map("{:.0%}".format)
runs["avg_tool_calls"] = runs["avg_tool_calls"].round(1)
runs["avg_turn_latency_s"] = runs["avg_turn_latency_s"].round(2)
runs["avg_tokens"] = runs["avg_tokens"].astype(int)
runs["avg_cost"] = runs["avg_cost"].map("${:.4f}".format)
runs["total_cost"] = runs["total_cost"].map("${:.4f}".format)
runs

,passed,questions,pass_rate,avg_tool_calls,avg_turn_latency_s,avg_tokens,avg_cost,total_cost
timestamp,,,,,,,,
2026-03-13 22:40,12,14,86%,5.9,NaN,44933,$0.0099,$0.1390
2026-03-14 07:42,12,14,86%,6.9,NaN,56346,$0.0170,$0.2386
2026-03-15 08:13,12,14,86%,13.6,NaN,80279,$0.0317,$0.4441
2026-03-15 15:05,13,14,93%,8.1,NaN,49770,$0.0151,$0.2119
2026-03-15 17:13,14,14,100%,6.0,NaN,23786,$0.0089,$0.1252
2026-03-15 17:38,13,14,93%,7.0,NaN,36117,$0.0135,$0.1888
2026-03-15 20:12,14,14,100%,5.5,NaN,28908,$0.0113,$0.1584
2026-03-16 09:41,14,15,93%,4.9,5.72,25900,$0.0104,$0.1564


In [10]:
# Select a run to inspect (default: latest)
RUN = results["timestamp"].max()

run = results[results["timestamp"] == RUN]
passed = run["passed"].sum()
total = len(run)
print(f"Selected: {RUN} | {passed}/{total} passed")

display_cols = ["workspace", "question", "passed", "tool_calls", "avg_turn_latency_s", "wall_time_s", "failed_assertions"]
run[display_cols].sort_values("passed")

Selected: 2026-03-16 09:41 | 14/15 passed


,workspace,question,passed,tool_calls,avg_turn_latency_s,wall_time_s,failed_assertions
109,sec-10k,Which company in the dataset had the highest total revenue?,False,2,4.16,16.1,contains 'amazon'; min_tool_calls >= 3
98,federalist-papers,What does Federalist No. 10 argue about factions?,True,6,3.80,38.4,NaN
99,federalist-papers,Which papers discuss the judiciary?,True,4,8.25,46.7,NaN
100,federalist-papers,What are the main themes across the Federalist Papers?,True,3,5.60,28.0,NaN
101,federalist-papers,What does Hamilton say about standing armies?,True,6,10.44,77.9,NaN
102,federalist-papers,How do Hamilton and Madison differ on federal power?,True,6,7.19,56.8,NaN
103,origin-of-species,What does Darwin say about natural selection in Chapter 4?,True,3,13.07,54.3,NaN
104,origin-of-species,How does Darwin explain the struggle for existence?,True,7,11.12,94.9,NaN
105,origin-of-species,What examples of variation under domestication does Darw...,True,13,2.06,37.3,NaN
106,sherlock-holmes,"What happens in ""A Scandal in Bohemia""?",True,4,3.17,18.7,NaN


In [11]:
# Failures for the selected run
failures = run[~run["passed"]]
if len(failures):
    print(f"{len(failures)} failures:")
    failures[["workspace", "question", "tool_calls", "failed_assertions", "error"]]
else:
    print("No failures!")

1 failures:


## Trace Viewer

In [12]:
# Build trace index: model → [(label, path), ...]
trace_index: dict[str, list[tuple[str, Path]]] = {}
for model_dir in sorted(TRACES_DIR.iterdir()):
    if not model_dir.is_dir() or model_dir.name.endswith(".csv"):
        continue
    entries = []
    for ws_dir in sorted(model_dir.iterdir()):
        if not ws_dir.is_dir():
            continue
        for f in sorted(ws_dir.glob("*.json")):
            data = json.loads(f.read_text())
            status = "PASS" if data["passed"] else "FAIL"
            entries.append((f"[{status}] {ws_dir.name} / {f.stem}", f))
    if entries:
        trace_index[model_dir.name] = entries

model_dropdown = widgets.Dropdown(
    options=list(trace_index.keys()),
    value=list(trace_index.keys())[-1],
    description="Model:",
    style={"description_width": "auto"},
    layout=widgets.Layout(width="60%"),
)
question_dropdown = widgets.Dropdown(
    description="Question:",
    style={"description_width": "auto"},
    layout=widgets.Layout(width="90%"),
)
output = widgets.Output()


def _update_questions(_change=None):
    entries = trace_index[model_dropdown.value]
    question_dropdown.options = [label for label, _ in entries]
    question_dropdown.value = question_dropdown.options[0]


def _show_trace(_change=None):
    entries = trace_index[model_dropdown.value]
    label_to_path = dict(entries)
    path = label_to_path.get(question_dropdown.value)
    if not path:
        return
    data = json.loads(path.read_text())
    with output:
        output.clear_output(wait=True)
        show_trace(data, max_chars=2000)


model_dropdown.observe(_update_questions, names="value")
question_dropdown.observe(_show_trace, names="value")

_update_questions()
display(model_dropdown, question_dropdown, output)
_show_trace()

Dropdown(description='Model:', index=1, layout=Layout(width='60%'), options=('openrouter-deepseek-deepseek-v3-…

Dropdown(description='Question:', layout=Layout(width='90%'), options=('[PASS] federalist-papers / how-do-hami…

Output()